> **Work In Progress** -- This notebook is a preliminary version. Content may change before the session. Check back for updates.

# Session 2 -- Linear Regression & Gradient Descent

**B4990 -- Artificial Intelligence** | Universidad Nacional de Rio Negro

**Date:** Wednesday, March 4, 2026 | **Instructor:** Federico Tula Rovaletti

---

## What we will cover today

1. **Linear regression from scratch** -- derive MSE, compute gradients by hand, implement in NumPy
2. **Closed-form solution** -- the Normal Equation, when it works, when it breaks
3. **MSE and Maximum Likelihood** -- why MSE is the "right" loss under Gaussian noise
4. **Gradient descent variants** -- batch, stochastic, mini-batch. Visualize convergence.
5. **Momentum and advanced optimizers** -- SGD + momentum from scratch, preview of Adam
6. **Convergence analysis** -- Hessian eigenvalues, condition number, optimal learning rate
7. **Regularization preview** -- Ridge regression and why it fixes ill-conditioned problems
8. **PyTorch refactor** -- replace manual gradients with autograd. Introduce the canonical training loop.

By the end of this session, you will understand the optimization pattern that drives ALL of machine learning:

> **forward pass -> compute loss -> compute gradients -> update parameters**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy import stats
import time

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Plot style
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

---

# Part 1: Linear Regression from Scratch

## 1.1 Generate Synthetic Data

We create a simple dataset where the true relationship is:

$$y = 3x + 7 + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, 1)$$

Our goal: learn $w = 3$ and $b = 7$ from the data.

In [ ]:
# True parameters
w_true = 3.0
b_true = 7.0

# Generate data
N = 200
x = np.random.uniform(-3, 3, size=N)
noise = np.random.randn(N) * 1.0
y = w_true * x + b_true + noise

print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")
print(f"x range: [{x.min():.2f}, {x.max():.2f}]")
print(f"y range: [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
# Visualize the data
plt.scatter(x, y, alpha=0.5, s=20, label='Data points')
plt.plot([-3, 3], [w_true * -3 + b_true, w_true * 3 + b_true], 'r--', 
         linewidth=2, label=f'True: y = {w_true}x + {b_true}')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Data for Linear Regression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.2 The Model

A linear regression model predicts:

$$\hat{y} = wx + b$$

where:
- $w$ is the **weight** (slope)
- $b$ is the **bias** (intercept)

These are the **parameters** we want to learn from data.

In [ ]:
def predict(x, w, b):
    """Forward pass: compute predictions."""
    return w * x + b

## 1.3 The Loss Function: Mean Squared Error (MSE)

How do we measure how bad our predictions are? We use **Mean Squared Error**:

$$\mathcal{L}(w, b) = \text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

- Squaring ensures all errors are positive
- Larger errors are penalized more (quadratic penalty)
- Our goal: find $w$ and $b$ that **minimize** this loss

In [ ]:
def mse_loss(y_pred, y_true):
    """Compute Mean Squared Error."""
    return np.mean((y_pred - y_true) ** 2)

In [ ]:
# Let's see the loss for some random parameters
w_init, b_init = 0.0, 0.0
y_pred = predict(x, w_init, b_init)
loss = mse_loss(y_pred, y)
print(f"With w={w_init}, b={b_init}: MSE = {loss:.4f}")

# Compare with the true parameters
y_pred_true = predict(x, w_true, b_true)
loss_true = mse_loss(y_pred_true, y)
print(f"With w={w_true}, b={b_true}: MSE = {loss_true:.4f} (this is the irreducible noise)")

## 1.4 Deriving the Gradient of MSE

To minimize the loss, we need to know **which direction to move** $w$ and $b$. This is what the **gradient** tells us.

### Step-by-step derivation

Start with:
$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2 = \frac{1}{N} \sum_{i=1}^{N} (wx_i + b - y_i)^2$$

**Gradient with respect to $w$:**

$$\frac{\partial \mathcal{L}}{\partial w} = \frac{1}{N} \sum_{i=1}^{N} 2(wx_i + b - y_i) \cdot x_i = \frac{2}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i) \cdot x_i$$

**Gradient with respect to $b$:**

$$\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{N} \sum_{i=1}^{N} 2(wx_i + b - y_i) \cdot 1 = \frac{2}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)$$

In vectorized form:
$$\frac{\partial \mathcal{L}}{\partial w} = \frac{2}{N} \cdot \mathbf{(\hat{y} - y)}^T \mathbf{x}$$
$$\frac{\partial \mathcal{L}}{\partial b} = \frac{2}{N} \cdot \sum(\hat{y} - y)$$

The gradient points in the direction of **steepest ascent**. We move in the **opposite** direction to decrease the loss.

In [ ]:
def compute_gradients(x, y, w, b):
    """Compute gradients of MSE loss w.r.t. w and b."""
    N = len(x)
    y_pred = predict(x, w, b)
    error = y_pred - y  # (N,)
    
    dw = (2.0 / N) * np.dot(error, x)
    db = (2.0 / N) * np.sum(error)
    
    return dw, db

In [ ]:
# Test: compute gradients at the initial point
dw, db = compute_gradients(x, y, w_init, b_init)
print(f"At w={w_init}, b={b_init}:")
print(f"  dL/dw = {dw:.4f}  (should be negative, pushing w toward 3)")
print(f"  dL/db = {db:.4f}  (should be negative, pushing b toward 7)")

## 1.5 Training: Batch Gradient Descent

The **gradient descent update rule** is:

$$w \leftarrow w - \alpha \frac{\partial \mathcal{L}}{\partial w}$$
$$b \leftarrow b - \alpha \frac{\partial \mathcal{L}}{\partial b}$$

where $\alpha$ is the **learning rate** -- how big a step we take.

In [ ]:
# Hyperparameters
lr = 0.01
n_epochs = 200

# Initialize parameters
w = 0.0
b = 0.0

# Track history
loss_history = []
w_history = [w]
b_history = [b]

# Training loop
for epoch in range(n_epochs):
    # Forward pass
    y_pred = predict(x, w, b)
    
    # Compute loss
    loss = mse_loss(y_pred, y)
    loss_history.append(loss)
    
    # Compute gradients
    dw, db = compute_gradients(x, y, w, b)
    
    # Update parameters
    w = w - lr * dw
    b = b - lr * db
    
    w_history.append(w)
    b_history.append(b)
    
    if epoch % 50 == 0 or epoch == n_epochs - 1:
        print(f"Epoch {epoch:3d}: loss = {loss:.4f}, w = {w:.4f}, b = {b:.4f}")

print(f"\nFinal: w = {w:.4f} (true: {w_true}), b = {b:.4f} (true: {b_true})")

In [ ]:
# Plot the loss curve
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Curve (Batch Gradient Descent)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot the fitted line
plt.scatter(x, y, alpha=0.5, s=20, label='Data')
x_line = np.linspace(-3, 3, 100)
plt.plot(x_line, w_true * x_line + b_true, 'r--', linewidth=2, label=f'True: y = {w_true}x + {b_true}')
plt.plot(x_line, w * x_line + b, 'g-', linewidth=2, label=f'Learned: y = {w:.2f}x + {b:.2f}')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Fitted Line vs True Line')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.6 Visualizing the Loss Landscape

The loss is a function of two variables: $w$ and $b$. We can plot it as a surface or contour plot.

In [ ]:
# Compute loss for a grid of (w, b) values
w_range = np.linspace(-2, 8, 100)
b_range = np.linspace(-2, 14, 100)
W_grid, B_grid = np.meshgrid(w_range, b_range)
L_grid = np.zeros_like(W_grid)

for i in range(W_grid.shape[0]):
    for j in range(W_grid.shape[1]):
        y_pred = predict(x, W_grid[i, j], B_grid[i, j])
        L_grid[i, j] = mse_loss(y_pred, y)

print(f"Loss grid shape: {L_grid.shape}")
print(f"Min loss on grid: {L_grid.min():.4f}")

In [ ]:
# Contour plot with GD path
fig, ax = plt.subplots(figsize=(9, 7))
contour = ax.contour(W_grid, B_grid, L_grid, levels=30, cmap='viridis')
ax.clabel(contour, inline=True, fontsize=8)

# Plot GD trajectory
ax.plot(w_history, b_history, 'r.-', markersize=3, linewidth=1, label='GD path')
ax.plot(w_history[0], b_history[0], 'ro', markersize=10, label='Start')
ax.plot(w_history[-1], b_history[-1], 'r*', markersize=15, label='End')
ax.plot(w_true, b_true, 'kx', markersize=12, markeredgewidth=3, label='True (w, b)')

ax.set_xlabel('w')
ax.set_ylabel('b')
ax.set_title('Loss Landscape with Gradient Descent Path')
ax.legend()
plt.tight_layout()
plt.show()

---

# Part 2: The Closed-Form Solution (Normal Equation)

Gradient descent is iterative -- it takes many steps to converge. But for linear regression, there is a **direct analytical solution**. We can solve for the optimal weights in one shot.

## 2.1 Derivation

We rewrite the model in matrix form. Define the **design matrix** $X$ where each row is a data point augmented with a 1 for the bias:

$$X = \begin{bmatrix} x_1 & 1 \\ x_2 & 1 \\ \vdots & \vdots \\ x_N & 1 \end{bmatrix} \in \mathbb{R}^{N \times 2}, \quad \mathbf{w} = \begin{bmatrix} w \\ b \end{bmatrix} \in \mathbb{R}^{2}$$

Then the predictions are $\hat{\mathbf{y}} = X\mathbf{w}$ and the MSE loss (ignoring the $\frac{1}{N}$ constant) is:

$$\mathcal{L}(\mathbf{w}) = \frac{1}{N}(X\mathbf{w} - \mathbf{y})^T(X\mathbf{w} - \mathbf{y})$$

### Full derivation

Expand the loss:

$$\mathcal{L}(\mathbf{w}) = \frac{1}{N}\left(\mathbf{w}^TX^TX\mathbf{w} - 2\mathbf{w}^TX^T\mathbf{y} + \mathbf{y}^T\mathbf{y}\right)$$

Take the gradient with respect to $\mathbf{w}$ and set it to zero:

$$\nabla_{\mathbf{w}} \mathcal{L} = \frac{1}{N}\left(2X^TX\mathbf{w} - 2X^T\mathbf{y}\right) = \mathbf{0}$$

$$X^TX\mathbf{w} = X^T\mathbf{y}$$

$$\boxed{\hat{\mathbf{w}} = (X^TX)^{-1}X^T\mathbf{y}}$$

This is the **Normal Equation**. It gives us the optimal weights directly -- no iteration needed.

### Why `np.linalg.solve` instead of `np.linalg.inv`?

In code, **never** write `np.linalg.inv(X.T @ X) @ X.T @ y`. Instead, use:

```python
w = np.linalg.solve(X.T @ X, X.T @ y)
```

Why? `solve` uses LU decomposition, which is:
- **More numerically stable** -- computing the inverse amplifies floating-point errors
- **Faster** -- $O(p^3)$ for solve vs $O(p^3)$ for inverse followed by a matrix multiply
- **The standard practice** in numerical linear algebra -- you almost never need the actual inverse matrix

In [ ]:
# Build the design matrix: [x, 1]
X_design = np.column_stack([x, np.ones(N)])
print(f"Design matrix shape: {X_design.shape}")
print(f"First 5 rows:\n{X_design[:5]}")

In [ ]:
# Normal equation: w = (X^T X)^{-1} X^T y
# Using np.linalg.solve (preferred)
w_closed = np.linalg.solve(X_design.T @ X_design, X_design.T @ y)

print(f"Closed-form solution:")
print(f"  w = {w_closed[0]:.6f} (true: {w_true})")
print(f"  b = {w_closed[1]:.6f} (true: {b_true})")
print(f"\nGD solution (after 200 epochs):")
print(f"  w = {w:.6f}")
print(f"  b = {b:.6f}")
print(f"\nDifference:")
print(f"  |w_gd - w_closed| = {abs(w - w_closed[0]):.6f}")
print(f"  |b_gd - b_closed| = {abs(b - w_closed[1]):.6f}")
print("\nGD is close but has not fully converged. With more epochs or a better lr, it would match.")

In [ ]:
# Also compare with np.linalg.inv (to show it gives the same answer but is less stable)
w_inv = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y

print(f"Using solve:  w = {w_closed[0]:.10f}, b = {w_closed[1]:.10f}")
print(f"Using inv:    w = {w_inv[0]:.10f}, b = {w_inv[1]:.10f}")
print(f"Difference:   {np.max(np.abs(w_closed - w_inv)):.2e}")
print("\nFor this well-conditioned problem, both give the same answer.")
print("The difference matters when X^T X is nearly singular (we will see this shortly).")

## 2.2 When Does the Closed-Form Fail?

The normal equation requires $(X^TX)^{-1}$ to exist. This fails when:

1. **$X^TX$ is singular** -- happens with perfectly collinear features (e.g., $x_2 = 2x_1$)
2. **$X^TX$ is near-singular** -- happens with highly correlated features (multicollinearity). The inverse exists but is numerically unstable.
3. **Computational cost** -- $(X^TX)^{-1}$ is $O(p^3)$ where $p$ is the number of features. For $p = 100{,}000$, this is infeasible. GD scales as $O(Np)$ per step.
4. **Regularization** -- the normal equation gives the unregularized solution. To add regularization, you need to modify the equation (we will see Ridge regression later in this notebook).

### Multicollinearity demo

Let's create features that are nearly perfectly correlated and watch the normal equation break.

In [ ]:
np.random.seed(42)

# Create a dataset with multicollinearity
N_mc = 100
x1 = np.random.randn(N_mc)
x2 = x1 + np.random.randn(N_mc) * 0.001  # x2 is almost exactly x1

print(f"Correlation between x1 and x2: {np.corrcoef(x1, x2)[0, 1]:.8f}")

# True model: y = 2*x1 + 3*x2 + 5 + noise
y_mc = 2 * x1 + 3 * x2 + 5 + np.random.randn(N_mc) * 0.5

# Build design matrix
X_mc = np.column_stack([x1, x2, np.ones(N_mc)])

# Check condition number of X^T X
XtX = X_mc.T @ X_mc
eigenvalues = np.linalg.eigvalsh(XtX)
condition_number = eigenvalues[-1] / eigenvalues[0]

print(f"\nEigenvalues of X^T X: {eigenvalues}")
print(f"Condition number: {condition_number:.2e}")
print("A condition number > 10^10 means the matrix is effectively singular.")

In [ ]:
# Try the normal equation -- it gives wildly wrong individual coefficients
try:
    w_mc_closed = np.linalg.solve(XtX, X_mc.T @ y_mc)
    print(f"Normal equation solution:")
    print(f"  w1 = {w_mc_closed[0]:.4f} (true: 2.0)")
    print(f"  w2 = {w_mc_closed[1]:.4f} (true: 3.0)")
    print(f"  b  = {w_mc_closed[2]:.4f} (true: 5.0)")
    print(f"\nThe individual weights are GARBAGE -- but notice:")
    print(f"  w1 + w2 = {w_mc_closed[0] + w_mc_closed[1]:.4f} (true: 2 + 3 = 5)")
    print("  The SUM is correct! The model cannot separate the effect of x1 vs x2")
    print("  because they carry nearly identical information.")
except np.linalg.LinAlgError as e:
    print(f"Normal equation FAILED: {e}")

# Predictions are still fine
y_pred_mc = X_mc @ w_mc_closed
print(f"\nMSE of predictions: {np.mean((y_pred_mc - y_mc)**2):.6f}")
print("Predictions are fine -- the problem is interpretability of individual weights.")

In [ ]:
# Now try GD on the same problem -- it works but converges slowly
# We use the multivariate version
def train_multivariate_gd(X, y, lr=0.001, n_epochs=1000):
    """Gradient descent for multivariate linear regression."""
    N, p = X.shape
    w = np.zeros(p)
    losses = []
    
    for epoch in range(n_epochs):
        y_pred = X @ w
        error = y_pred - y
        loss = np.mean(error ** 2)
        losses.append(loss)
        
        grad = (2.0 / N) * (X.T @ error)
        w = w - lr * grad
    
    return w, losses

w_mc_gd, losses_mc = train_multivariate_gd(X_mc, y_mc, lr=0.01, n_epochs=2000)

print(f"GD solution (2000 epochs):")
print(f"  w1 = {w_mc_gd[0]:.4f}")
print(f"  w2 = {w_mc_gd[1]:.4f}")
print(f"  b  = {w_mc_gd[2]:.4f}")
print(f"  w1 + w2 = {w_mc_gd[0] + w_mc_gd[1]:.4f}")
print(f"\nGD converges to a solution with similar prediction quality")
print(f"  MSE: {losses_mc[-1]:.6f}")
print(f"\nBut GD tends to find a more 'balanced' solution due to implicit regularization")
print(f"  (it does not overshoot individual weights as wildly as the normal equation)")

---

# Part 3: MSE and Maximum Likelihood -- Why MSE is the "Right" Loss

You might wonder: *why MSE specifically?* Why not mean absolute error, or mean cubed error?

The answer comes from probability theory. MSE is not an arbitrary choice -- it falls out naturally when you assume the noise in your data is **Gaussian**.

## 3.1 The Probabilistic Model

Assume each data point follows:

$$y_i = \mathbf{w}^T \mathbf{x}_i + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)$$

This means $y_i | \mathbf{x}_i \sim \mathcal{N}(\mathbf{w}^T \mathbf{x}_i, \sigma^2)$. The probability of observing a specific $y_i$ given $\mathbf{x}_i$ is:

$$p(y_i | \mathbf{x}_i, \mathbf{w}) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - \mathbf{w}^T\mathbf{x}_i)^2}{2\sigma^2}\right)$$

## 3.2 The Likelihood Function

Assuming data points are independent, the **likelihood** of observing all our data is:

$$\mathcal{L}(\mathbf{w}) = \prod_{i=1}^{N} p(y_i | \mathbf{x}_i, \mathbf{w}) = \prod_{i=1}^{N} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - \mathbf{w}^T\mathbf{x}_i)^2}{2\sigma^2}\right)$$

## 3.3 Log-Likelihood

Products are hard to optimize. Take the log:

$$\log \mathcal{L}(\mathbf{w}) = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{N}(y_i - \mathbf{w}^T\mathbf{x}_i)^2$$

## 3.4 The Punchline

To **maximize** the log-likelihood, we need to **minimize**:

$$\sum_{i=1}^{N}(y_i - \mathbf{w}^T\mathbf{x}_i)^2$$

which is exactly the MSE (up to a constant factor $N$).

$$\boxed{\text{Maximum Likelihood under Gaussian noise} \iff \text{Minimize MSE}}$$

### What if the noise is not Gaussian?

| Noise Distribution | Corresponding Loss | When to use |
|---|---|---|
| Gaussian $\mathcal{N}(0, \sigma^2)$ | MSE (L2 loss) | Default for regression |
| Laplace | MAE (L1 loss) | Data with outliers |
| Bernoulli | Cross-entropy | Binary classification |

**Different assumptions about the data-generating process lead to different loss functions.** MSE is not universal -- it is the right choice when errors are roughly symmetric and bell-shaped.

In [ ]:
# Let's check: are our residuals actually Gaussian?
# Use the closed-form solution for the best fit
y_pred_best = X_design @ w_closed
residuals = y - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of residuals
axes[0].hist(residuals, bins=25, density=True, alpha=0.7, color='steelblue', edgecolor='black')
# Overlay a Gaussian
x_gauss = np.linspace(-4, 4, 200)
axes[0].plot(x_gauss, stats.norm.pdf(x_gauss, loc=np.mean(residuals), scale=np.std(residuals)),
             'r-', linewidth=2, label=f'N({np.mean(residuals):.2f}, {np.std(residuals):.2f}$^2$)')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Density')
axes[0].set_title('Histogram of Residuals')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# QQ plot
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Shapiro-Wilk test for normality
stat, p_value = stats.shapiro(residuals)
print(f"Shapiro-Wilk test: statistic = {stat:.4f}, p-value = {p_value:.4f}")
print(f"p > 0.05 => we do NOT reject the null hypothesis that residuals are Gaussian.")
print(f"This confirms: MSE was the correct loss function for this data.")

---

# Part 4: Gradient Descent Variants

So far we used **Batch Gradient Descent** -- we compute the gradient using ALL N samples at every step.

What if N is 10 million? Each step would be very expensive.

Two alternatives:
- **Stochastic Gradient Descent (SGD):** Use **1 random sample** per step
- **Mini-batch Gradient Descent:** Use **B random samples** per step (B is the batch size)

| Variant | Samples/step | Speed/step | Noise | Memory |
|---------|:---:|:---:|:---:|:---:|
| Batch | N | Slow | Low | High |
| SGD | 1 | Fast | High | Low |
| Mini-batch | B | Medium | Medium | Medium |

## 4.1 Implement All Three Variants

In [ ]:
def train_batch_gd(x, y, lr=0.01, n_epochs=200):
    """Batch Gradient Descent: use ALL samples per step."""
    w, b = 0.0, 0.0
    loss_history = []
    w_history, b_history = [w], [b]
    
    for epoch in range(n_epochs):
        y_pred = predict(x, w, b)
        loss = mse_loss(y_pred, y)
        loss_history.append(loss)
        
        dw, db = compute_gradients(x, y, w, b)
        w -= lr * dw
        b -= lr * db
        
        w_history.append(w)
        b_history.append(b)
    
    return w, b, loss_history, w_history, b_history


def train_sgd(x, y, lr=0.01, n_epochs=200):
    """Stochastic Gradient Descent: use 1 random sample per step."""
    w, b = 0.0, 0.0
    loss_history = []
    w_history, b_history = [w], [b]
    N = len(x)
    
    for epoch in range(n_epochs):
        # Shuffle indices each epoch
        indices = np.random.permutation(N)
        
        for i in indices:
            # One sample at a time
            xi = x[i:i+1]
            yi = y[i:i+1]
            
            y_pred = predict(xi, w, b)
            error = y_pred - yi
            
            dw = 2.0 * np.dot(error, xi)
            db = 2.0 * np.sum(error)
            
            w -= lr * dw
            b -= lr * db
        
        # Record loss on full dataset at end of each epoch
        full_loss = mse_loss(predict(x, w, b), y)
        loss_history.append(full_loss)
        w_history.append(w)
        b_history.append(b)
    
    return w, b, loss_history, w_history, b_history


def train_minibatch_gd(x, y, lr=0.01, n_epochs=200, batch_size=32):
    """Mini-batch Gradient Descent: use B random samples per step."""
    w, b = 0.0, 0.0
    loss_history = []
    w_history, b_history = [w], [b]
    N = len(x)
    
    for epoch in range(n_epochs):
        indices = np.random.permutation(N)
        
        for start in range(0, N, batch_size):
            batch_idx = indices[start:start + batch_size]
            x_batch = x[batch_idx]
            y_batch = y[batch_idx]
            
            y_pred = predict(x_batch, w, b)
            error = y_pred - y_batch
            B = len(x_batch)
            
            dw = (2.0 / B) * np.dot(error, x_batch)
            db = (2.0 / B) * np.sum(error)
            
            w -= lr * dw
            b -= lr * db
        
        full_loss = mse_loss(predict(x, w, b), y)
        loss_history.append(full_loss)
        w_history.append(w)
        b_history.append(b)
    
    return w, b, loss_history, w_history, b_history

## 4.2 Compare Convergence

In [ ]:
# Run all three variants
np.random.seed(42)
w_batch, b_batch, loss_batch, wh_batch, bh_batch = train_batch_gd(x, y, lr=0.01, n_epochs=100)

np.random.seed(42)
w_sgd, b_sgd, loss_sgd, wh_sgd, bh_sgd = train_sgd(x, y, lr=0.001, n_epochs=100)

np.random.seed(42)
w_mb, b_mb, loss_mb, wh_mb, bh_mb = train_minibatch_gd(x, y, lr=0.005, n_epochs=100, batch_size=32)

print(f"Batch GD:     w = {w_batch:.4f}, b = {b_batch:.4f}")
print(f"SGD:          w = {w_sgd:.4f}, b = {b_sgd:.4f}")
print(f"Mini-batch:   w = {w_mb:.4f}, b = {b_mb:.4f}")
print(f"True:         w = {w_true:.4f}, b = {b_true:.4f}")

In [ ]:
# Plot loss curves side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(loss_batch, 'b-')
axes[0].set_title('Batch GD')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_sgd, 'r-')
axes[1].set_title('SGD')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(loss_mb, 'g-')
axes[2].set_title('Mini-batch GD (B=32)')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.set_ylim(bottom=0)

plt.suptitle('Loss Curves: GD Variants', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# All three on the same plot
plt.plot(loss_batch, 'b-', label='Batch GD', linewidth=2)
plt.plot(loss_sgd, 'r-', label='SGD', alpha=0.7)
plt.plot(loss_mb, 'g-', label='Mini-batch (B=32)', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Convergence Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.3 Optimization Trajectories on the Loss Landscape

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titles = ['Batch GD', 'SGD', 'Mini-batch GD']
histories = [
    (wh_batch, bh_batch),
    (wh_sgd, bh_sgd),
    (wh_mb, bh_mb),
]
colors = ['blue', 'red', 'green']

for ax, title, (wh, bh), color in zip(axes, titles, histories, colors):
    ax.contour(W_grid, B_grid, L_grid, levels=30, cmap='viridis', alpha=0.5)
    ax.plot(wh, bh, '.-', color=color, markersize=2, linewidth=0.8, alpha=0.7)
    ax.plot(wh[0], bh[0], 'o', color=color, markersize=8)
    ax.plot(wh[-1], bh[-1], '*', color=color, markersize=12)
    ax.plot(w_true, b_true, 'kx', markersize=12, markeredgewidth=3)
    ax.set_xlabel('w')
    ax.set_ylabel('b')
    ax.set_title(title)

plt.suptitle('Optimization Trajectories on Loss Landscape', fontsize=14)
plt.tight_layout()
plt.show()

Notice how:
- **Batch GD** takes a smooth, direct path to the minimum
- **SGD** zigzags because each single-sample gradient is noisy
- **Mini-batch GD** is a compromise: smoother than SGD, more flexible than batch

## 4.4 Learning Rate Experiment

The learning rate $\alpha$ controls the step size. What happens when it is too small, just right, or too large?

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 1.0]
lr_results = {}

for lr_val in learning_rates:
    try:
        _, _, losses, _, _ = train_batch_gd(x, y, lr=lr_val, n_epochs=100)
        lr_results[lr_val] = losses
    except (OverflowError, FloatingPointError):
        lr_results[lr_val] = [float('inf')] * 100

# Plot
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, lr_val in zip(axes, learning_rates):
    losses = lr_results[lr_val]
    ax.plot(losses, linewidth=2)
    ax.set_title(f'lr = {lr_val}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.grid(True, alpha=0.3)
    # Cap y-axis for divergent cases
    if any(l > 1e6 for l in losses if np.isfinite(l)):
        ax.set_ylim(0, max(100, losses[0] * 2))
        ax.text(0.5, 0.9, 'DIVERGES!', transform=ax.transAxes,
                ha='center', fontsize=14, color='red', fontweight='bold')

plt.suptitle('Effect of Learning Rate on Convergence', fontsize=14)
plt.tight_layout()
plt.show()

**Key observations:**
- **lr = 0.001:** Converges, but very slowly. We would need many more epochs.
- **lr = 0.01:** Good convergence. A reasonable default for this problem.
- **lr = 0.1:** Fast convergence, but may oscillate near the minimum.
- **lr = 1.0:** DIVERGES. The steps are so large that we overshoot the minimum and the loss explodes.

In practice, learning rate is the **most important hyperparameter** to tune.

---

# Part 5: Momentum and Advanced Optimizers

## 5.1 The Problem with Vanilla SGD

Vanilla SGD can be slow in certain loss landscapes:
- It oscillates in directions of high curvature
- It makes slow progress in directions of low curvature
- Ravines (elongated valleys) are particularly problematic

## 5.2 SGD with Momentum

**Intuition:** Think of a ball rolling downhill. It does not stop and change direction instantly -- it has *momentum*. Past gradients contribute to the current velocity.

The update rule with momentum:

$$v_t = \beta \cdot v_{t-1} + \nabla \mathcal{L}(\mathbf{w}_t)$$
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \cdot v_t$$

where:
- $v_t$ is the **velocity** (exponential moving average of past gradients)
- $\beta$ is the **momentum coefficient** (typically 0.9)
- The velocity accumulates gradients: if the gradient consistently points in one direction, the velocity builds up and we move faster
- If the gradient oscillates (changes sign), the velocity cancels out the oscillations

## 5.3 Nesterov Momentum (brief)

A subtle improvement: instead of computing the gradient at the current position, compute it at the **lookahead** position $\mathbf{w}_t - \alpha \beta v_{t-1}$:

$$v_t = \beta \cdot v_{t-1} + \nabla \mathcal{L}(\mathbf{w}_t - \alpha \beta v_{t-1})$$
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \cdot v_t$$

The idea: "look before you leap." Compute the gradient at where you *will* be, not where you are now. This leads to faster convergence in practice.

In [ ]:
def train_sgd_momentum(x, y, lr=0.01, n_epochs=200, momentum=0.9, batch_size=32):
    """SGD with momentum, from scratch."""
    w, b = 0.0, 0.0
    v_w, v_b = 0.0, 0.0  # velocity
    loss_history = []
    w_history, b_history = [w], [b]
    N = len(x)
    
    for epoch in range(n_epochs):
        indices = np.random.permutation(N)
        
        for start in range(0, N, batch_size):
            batch_idx = indices[start:start + batch_size]
            x_batch = x[batch_idx]
            y_batch = y[batch_idx]
            B = len(x_batch)
            
            y_pred = predict(x_batch, w, b)
            error = y_pred - y_batch
            
            dw = (2.0 / B) * np.dot(error, x_batch)
            db = (2.0 / B) * np.sum(error)
            
            # Momentum update
            v_w = momentum * v_w + dw
            v_b = momentum * v_b + db
            
            w -= lr * v_w
            b -= lr * v_b
        
        full_loss = mse_loss(predict(x, w, b), y)
        loss_history.append(full_loss)
        w_history.append(w)
        b_history.append(b)
    
    return w, b, loss_history, w_history, b_history

In [ ]:
# Compare SGD vs SGD + momentum
np.random.seed(42)
w_mb2, b_mb2, loss_mb2, wh_mb2, bh_mb2 = train_minibatch_gd(
    x, y, lr=0.005, n_epochs=100, batch_size=32)

np.random.seed(42)
w_mom, b_mom, loss_mom, wh_mom, bh_mom = train_sgd_momentum(
    x, y, lr=0.005, n_epochs=100, momentum=0.9, batch_size=32)

print(f"Mini-batch SGD:              w = {w_mb2:.4f}, b = {b_mb2:.4f}, final loss = {loss_mb2[-1]:.6f}")
print(f"Mini-batch SGD + momentum:   w = {w_mom:.4f}, b = {b_mom:.4f}, final loss = {loss_mom[-1]:.6f}")

In [ ]:
# Side-by-side: loss curves and trajectories
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(loss_mb2, 'b-', label='SGD (no momentum)', linewidth=2)
axes[0].plot(loss_mom, 'r-', label='SGD + momentum (0.9)', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Trajectory: SGD
axes[1].contour(W_grid, B_grid, L_grid, levels=30, cmap='viridis', alpha=0.5)
axes[1].plot(wh_mb2, bh_mb2, 'b.-', markersize=3, linewidth=0.8, alpha=0.7)
axes[1].plot(wh_mb2[0], bh_mb2[0], 'bo', markersize=8)
axes[1].plot(wh_mb2[-1], bh_mb2[-1], 'b*', markersize=12)
axes[1].plot(w_true, b_true, 'kx', markersize=12, markeredgewidth=3)
axes[1].set_xlabel('w')
axes[1].set_ylabel('b')
axes[1].set_title('SGD (no momentum)')

# Trajectory: SGD + momentum
axes[2].contour(W_grid, B_grid, L_grid, levels=30, cmap='viridis', alpha=0.5)
axes[2].plot(wh_mom, bh_mom, 'r.-', markersize=3, linewidth=0.8, alpha=0.7)
axes[2].plot(wh_mom[0], bh_mom[0], 'ro', markersize=8)
axes[2].plot(wh_mom[-1], bh_mom[-1], 'r*', markersize=12)
axes[2].plot(w_true, b_true, 'kx', markersize=12, markeredgewidth=3)
axes[2].set_xlabel('w')
axes[2].set_ylabel('b')
axes[2].set_title('SGD + momentum (0.9)')

plt.suptitle('SGD vs SGD + Momentum', fontsize=14)
plt.tight_layout()
plt.show()

## 5.4 Preview: Adam Optimizer

**Adam** (Adaptive Moment Estimation) combines two ideas:
1. **Momentum** (first moment) -- like what we just implemented
2. **Adaptive learning rates** (second moment) -- each parameter gets its own learning rate, based on how much its gradient has varied

We will not implement Adam from scratch today, but here is the intuition:
- Parameters with consistently large gradients get smaller learning rates (to prevent overshooting)
- Parameters with small, infrequent gradients get larger learning rates (to make faster progress)
- It also includes bias correction for the initial steps

Adam is the **default optimizer** in most deep learning work. You will use `torch.optim.Adam` starting in Session 8. For now, SGD with momentum is sufficient and teaches you the core ideas.

---

# Part 6: Convergence Analysis

So far we picked learning rates by trial and error. Can we be more principled? Yes -- the theory of convex optimization gives us precise answers for quadratic losses.

## 6.1 The Hessian Matrix

The **Hessian** $H$ is the matrix of second derivatives of the loss. For MSE with design matrix $X$:

$$H = \frac{2}{N} X^T X$$

The Hessian captures the **curvature** of the loss landscape:
- Eigenvalues of $H$ tell us how "steep" the loss is in each direction
- Large eigenvalue = steep direction (loss changes rapidly)
- Small eigenvalue = flat direction (loss changes slowly)

## 6.2 Optimal Learning Rate

For gradient descent on a quadratic loss to converge, the learning rate must satisfy:

$$\alpha < \frac{2}{\lambda_{\max}}$$

where $\lambda_{\max}$ is the largest eigenvalue of the Hessian. If $\alpha \geq \frac{2}{\lambda_{\max}}$, the optimization **diverges**.

The optimal learning rate (fastest convergence) is:

$$\alpha^* = \frac{2}{\lambda_{\max} + \lambda_{\min}}$$

## 6.3 Condition Number

The **condition number** of the Hessian is:

$$\kappa = \frac{\lambda_{\max}}{\lambda_{\min}}$$

- $\kappa = 1$: perfectly conditioned. All directions have the same curvature. GD converges fast.
- $\kappa \gg 1$: ill-conditioned. Some directions are much steeper than others. GD oscillates in steep directions and creeps along flat directions. Convergence is slow.

The convergence rate of GD is:

$$\text{error after } t \text{ steps} \leq \left(\frac{\kappa - 1}{\kappa + 1}\right)^{2t} \cdot \text{initial error}$$

High $\kappa$ means the ratio $\frac{\kappa-1}{\kappa+1}$ is close to 1, so convergence is slow.

In [ ]:
# Compute the Hessian for our original 1D problem
H = (2.0 / N) * (X_design.T @ X_design)
eigenvalues_H = np.linalg.eigvalsh(H)

print(f"Hessian H = (2/N) * X^T X:")
print(f"{H}")
print(f"\nEigenvalues: {eigenvalues_H}")
print(f"Condition number: {eigenvalues_H[-1] / eigenvalues_H[0]:.4f}")
print(f"\nMax safe learning rate: lr < 2 / lambda_max = {2.0 / eigenvalues_H[-1]:.4f}")
print(f"Optimal learning rate:  lr* = 2 / (lambda_max + lambda_min) = {2.0 / (eigenvalues_H[-1] + eigenvalues_H[0]):.4f}")

In [ ]:
# Demo: lr just below threshold => converges. lr above threshold => diverges.
lr_max = 2.0 / eigenvalues_H[-1]
lr_optimal = 2.0 / (eigenvalues_H[-1] + eigenvalues_H[0])

test_lrs = {
    'lr = 0.01 (conservative)': 0.01,
    f'lr = {lr_optimal:.4f} (optimal)': lr_optimal,
    f'lr = {lr_max * 0.95:.4f} (95% of max)': lr_max * 0.95,
    f'lr = {lr_max * 1.05:.4f} (105% of max -- DIVERGES)': lr_max * 1.05,
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (label, lr_val) in zip(axes, test_lrs.items()):
    _, _, losses, _, _ = train_batch_gd(x, y, lr=lr_val, n_epochs=50)
    ax.plot(losses, linewidth=2)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.grid(True, alpha=0.3)
    if any(l > 1e6 for l in losses if np.isfinite(l)):
        ax.set_ylim(0, max(100, losses[0] * 2))
        ax.text(0.5, 0.9, 'DIVERGES!', transform=ax.transAxes,
                ha='center', fontsize=14, color='red', fontweight='bold')

plt.suptitle('Learning Rate vs Hessian Eigenvalues', fontsize=14)
plt.tight_layout()
plt.show()

## 6.4 Ill-Conditioning and Feature Scaling

Now let's see why **feature scaling** matters. We create a problem where one feature has a much larger range than another, making the Hessian ill-conditioned.

In [ ]:
np.random.seed(42)

# Create an ill-conditioned problem
N_ill = 200
x1_ill = np.random.randn(N_ill) * 100   # large scale
x2_ill = np.random.randn(N_ill) * 0.01  # tiny scale
y_ill = 0.5 * x1_ill + 200 * x2_ill + 3 + np.random.randn(N_ill) * 0.5

X_ill = np.column_stack([x1_ill, x2_ill, np.ones(N_ill)])

# Compute Hessian and condition number
H_ill = (2.0 / N_ill) * (X_ill.T @ X_ill)
eig_ill = np.linalg.eigvalsh(H_ill)
kappa_ill = eig_ill[-1] / eig_ill[0]

print(f"UNSCALED features:")
print(f"  Eigenvalues of Hessian: {eig_ill}")
print(f"  Condition number: {kappa_ill:.2e}")
print(f"  Max safe lr: {2.0 / eig_ill[-1]:.6f}")

# Now scale the features
x1_scaled = (x1_ill - x1_ill.mean()) / x1_ill.std()
x2_scaled = (x2_ill - x2_ill.mean()) / x2_ill.std()
X_scaled = np.column_stack([x1_scaled, x2_scaled, np.ones(N_ill)])

H_scaled = (2.0 / N_ill) * (X_scaled.T @ X_scaled)
eig_scaled = np.linalg.eigvalsh(H_scaled)
kappa_scaled = eig_scaled[-1] / eig_scaled[0]

print(f"\nSCALED features (zero mean, unit variance):")
print(f"  Eigenvalues of Hessian: {eig_scaled}")
print(f"  Condition number: {kappa_scaled:.2e}")
print(f"  Max safe lr: {2.0 / eig_scaled[-1]:.6f}")

print(f"\nScaling improved the condition number by a factor of {kappa_ill / kappa_scaled:.0f}x!")

In [ ]:
# Train on unscaled vs scaled and compare convergence
# Use a very small lr (forced by the unscaled condition number)
safe_lr_unscaled = 0.5 * (2.0 / eig_ill[-1])  # 50% of max safe lr for unscaled

w_unscaled, losses_unscaled = train_multivariate_gd(X_ill, y_ill, lr=safe_lr_unscaled, n_epochs=500)
w_scaled, losses_scaled = train_multivariate_gd(X_scaled, y_ill, lr=0.1, n_epochs=500)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses_unscaled, 'r-', label=f'Unscaled (kappa={kappa_ill:.1e}, lr={safe_lr_unscaled:.6f})', linewidth=2)
ax.plot(losses_scaled, 'b-', label=f'Scaled (kappa={kappa_scaled:.1f}, lr=0.1)', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Feature Scaling Dramatically Improves Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print("This is WHY feature scaling is a standard preprocessing step.")
print("It is not just a trick -- it directly improves the condition number of the Hessian.")

---

# Part 7: Regularization Preview -- Ridge Regression

We saw that multicollinearity makes the normal equation unstable. The fix is **regularization**: add a penalty that discourages large weights.

## 7.1 Ridge Regression (L2 Regularization)

Add a penalty term to the loss:

$$\mathcal{L}_{\text{ridge}}(\mathbf{w}) = \frac{1}{N}\|X\mathbf{w} - \mathbf{y}\|^2 + \lambda \|\mathbf{w}\|^2$$

where $\lambda > 0$ controls the strength of regularization.

The closed-form solution becomes:

$$\hat{\mathbf{w}}_{\text{ridge}} = (X^TX + N\lambda I)^{-1}X^T\mathbf{y}$$

Adding $N\lambda I$ to $X^TX$ makes it **always invertible**, even when $X^TX$ is singular! The regularization term shifts all eigenvalues away from zero.

We will study regularization in depth in Session 8. For now, let's see it fix the multicollinearity problem.

In [ ]:
# Ridge regression on the multicollinear dataset from Part 2
lambdas = [0, 1e-6, 1e-4, 1e-2, 1.0]

print(f"{'lambda':>10} | {'w1':>10} {'w2':>10} {'b':>10} | {'w1+w2':>10} | {'cond(X^TX + nLI)':>18}")
print("-" * 85)

for lam in lambdas:
    XtX_mc = X_mc.T @ X_mc
    regularized = XtX_mc + N_mc * lam * np.eye(3)
    w_ridge = np.linalg.solve(regularized, X_mc.T @ y_mc)
    cond = np.linalg.cond(regularized)
    print(f"{lam:10.1e} | {w_ridge[0]:10.4f} {w_ridge[1]:10.4f} {w_ridge[2]:10.4f} | {w_ridge[0]+w_ridge[1]:10.4f} | {cond:18.2e}")

print(f"\nTrue values: w1=2.0, w2=3.0, b=5.0, w1+w2=5.0")
print(f"\nNotice: as lambda increases, the condition number drops and the weights become more stable.")
print(f"Too much regularization (lambda=1.0) biases the weights toward zero.")
print(f"Finding the right lambda is a bias-variance tradeoff -- more in Session 8.")

In [ ]:
# Visualize the effect of L2 regularization on the loss landscape (2D, original problem)
# Show contours of MSE + lambda * ||w||^2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
lambda_vals = [0, 0.5, 2.0]

for ax, lam in zip(axes, lambda_vals):
    # Compute regularized loss
    L_reg = np.zeros_like(W_grid)
    for i in range(W_grid.shape[0]):
        for j in range(W_grid.shape[1]):
            y_p = predict(x, W_grid[i, j], B_grid[i, j])
            L_reg[i, j] = mse_loss(y_p, y) + lam * (W_grid[i, j]**2 + B_grid[i, j]**2)
    
    contour = ax.contour(W_grid, B_grid, L_reg, levels=30, cmap='viridis')
    ax.plot(w_true, b_true, 'kx', markersize=12, markeredgewidth=3, label='True (w,b)')
    
    # Show the L2 penalty circle
    theta = np.linspace(0, 2*np.pi, 100)
    for r in [1, 2, 3, 4]:
        ax.plot(r * np.cos(theta), r * np.sin(theta), 'r--', alpha=0.3, linewidth=0.8)
    
    # Find and plot the minimum
    min_idx = np.unravel_index(L_reg.argmin(), L_reg.shape)
    ax.plot(W_grid[min_idx], B_grid[min_idx], 'r*', markersize=15, label='Minimum')
    
    ax.set_xlabel('w')
    ax.set_ylabel('b')
    ax.set_title(f'lambda = {lam}')
    ax.legend(fontsize=9)

plt.suptitle('Effect of L2 Regularization on Loss Landscape', fontsize=14)
plt.tight_layout()
plt.show()

print("As lambda increases, the minimum shifts toward the origin (smaller weights).")
print("The L2 penalty 'pulls' the solution toward (0, 0).")
print("Red dashed circles show the L2 norm contours.")

---

# Part 8: PyTorch Refactor

We just spent significant effort computing gradients by hand. Now let's see how **PyTorch autograd** does it automatically.

The idea:
1. Define parameters with `requires_grad=True`
2. Compute the loss using normal operations
3. Call `loss.backward()` -- PyTorch computes ALL gradients automatically
4. Update parameters using `optimizer.step()`

**This is the last time you will compute gradients by hand.**

## 8.1 PyTorch Tensors and Autograd

In [ ]:
# Convert data to PyTorch tensors
x_t = torch.tensor(x, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

# Create learnable parameters -- requires_grad=True tells PyTorch to track gradients
w_t = torch.tensor(0.0, requires_grad=True)
b_t = torch.tensor(0.0, requires_grad=True)

print(f"x_t shape: {x_t.shape}, dtype: {x_t.dtype}")
print(f"w_t: {w_t}, requires_grad: {w_t.requires_grad}")
print(f"b_t: {b_t}, requires_grad: {b_t.requires_grad}")

In [ ]:
# Forward pass and backward pass -- let's see autograd in action
y_pred_t = w_t * x_t + b_t
loss_t = torch.mean((y_pred_t - y_t) ** 2)

print(f"Loss: {loss_t.item():.4f}")

# This one line replaces our entire compute_gradients function!
loss_t.backward()

print(f"\nPyTorch autograd gradients:")
print(f"  dL/dw = {w_t.grad.item():.6f}")
print(f"  dL/db = {b_t.grad.item():.6f}")

# Compare with our manual NumPy gradients
dw_np, db_np = compute_gradients(x, y, 0.0, 0.0)
print(f"\nNumPy manual gradients:")
print(f"  dL/dw = {dw_np:.6f}")
print(f"  dL/db = {db_np:.6f}")

print(f"\nDifference (should be ~0):")
print(f"  dw: {abs(w_t.grad.item() - dw_np):.8f}")
print(f"  db: {abs(b_t.grad.item() - db_np):.8f}")

The gradients match! `loss.backward()` computed the exact same derivatives we derived by hand.

But it works for **any differentiable computation** -- not just linear regression. For a neural network with millions of parameters, autograd handles all the chain rule bookkeeping automatically.

## 8.2 The Canonical PyTorch Training Loop

This is the pattern you will use for the rest of the course:

```python
for epoch in range(n_epochs):
    optimizer.zero_grad()    # 1. Clear old gradients
    y_pred = model(x)        # 2. Forward pass
    loss = loss_fn(y_pred, y)# 3. Compute loss
    loss.backward()          # 4. Compute gradients (autograd)
    optimizer.step()         # 5. Update parameters
```

**Critical:** `optimizer.zero_grad()` must come first. PyTorch **accumulates** gradients by default. If you forget to zero them, gradients from previous iterations add up and your training goes wrong.

In [ ]:
# Full PyTorch training loop
torch.manual_seed(42)

# Data
x_t = torch.tensor(x, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

# Parameters
w_t = torch.tensor(0.0, requires_grad=True)
b_t = torch.tensor(0.0, requires_grad=True)

# Optimizer
optimizer = torch.optim.SGD([w_t, b_t], lr=0.01)

# Training
n_epochs = 200
pt_loss_history = []

for epoch in range(n_epochs):
    # 1. Zero gradients
    optimizer.zero_grad()
    
    # 2. Forward pass
    y_pred = w_t * x_t + b_t
    
    # 3. Compute loss
    loss = torch.mean((y_pred - y_t) ** 2)
    
    # 4. Backward pass (compute gradients)
    loss.backward()
    
    # 5. Update parameters
    optimizer.step()
    
    pt_loss_history.append(loss.item())
    
    if epoch % 50 == 0 or epoch == n_epochs - 1:
        print(f"Epoch {epoch:3d}: loss = {loss.item():.4f}, "
              f"w = {w_t.item():.4f}, b = {b_t.item():.4f}")

print(f"\nFinal: w = {w_t.item():.4f} (true: {w_true}), b = {b_t.item():.4f} (true: {b_true})")

In [ ]:
# Compare NumPy and PyTorch loss curves
plt.plot(loss_history[:200], 'b-', label='NumPy (manual gradients)', linewidth=2)
plt.plot(pt_loss_history, 'r--', label='PyTorch (autograd)', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('NumPy vs PyTorch: Same Convergence')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The curves overlap perfectly. PyTorch autograd produces the **exact same optimization** as our manual gradient code. The only difference: we did not have to derive or implement any gradients.

## 8.3 EXERCISE: What Happens Without `zero_grad()`?

Run the training loop below. It is missing `optimizer.zero_grad()`. What goes wrong?

In [ ]:
# EXERCISE: Run this and observe what happens
# Then fix it by adding optimizer.zero_grad() in the right place

w_broken = torch.tensor(0.0, requires_grad=True)
b_broken = torch.tensor(0.0, requires_grad=True)
optimizer_broken = torch.optim.SGD([w_broken, b_broken], lr=0.01)

broken_losses = []
for epoch in range(50):
    # BUG: missing optimizer.zero_grad() here!
    y_pred = w_broken * x_t + b_broken
    loss = torch.mean((y_pred - y_t) ** 2)
    loss.backward()
    optimizer_broken.step()
    broken_losses.append(loss.item())

plt.plot(broken_losses, 'r-', label='Without zero_grad()', linewidth=2)
plt.plot(pt_loss_history[:50], 'b--', label='With zero_grad()', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('What happens without optimizer.zero_grad()?')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Without zero_grad: final w = {w_broken.item():.4f}, b = {b_broken.item():.4f}")
print("Gradients accumulate, making each update larger than intended!")

### SOLUTION

Without `zero_grad()`, gradients from every previous iteration accumulate. The effective gradient grows larger and larger, causing the optimizer to take increasingly wild steps. The fix is to add `optimizer.zero_grad()` as the first line inside the loop.

## 8.4 EXERCISE: Verify Gradient Match Across Iterations

Run both NumPy and PyTorch for 5 iterations with the same initial weights and learning rate. Print the gradients at each step and verify they match.

In [ ]:
# EXERCISE: Fill in the missing parts marked with ???
# Then verify the gradients match at each step

# --- SOLUTION ---

# NumPy version
w_np, b_np = 0.0, 0.0
lr_check = 0.01

# PyTorch version
w_pt = torch.tensor(0.0, requires_grad=True)
b_pt = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.SGD([w_pt, b_pt], lr=lr_check)

print(f"{'Step':>4} | {'NumPy dw':>12} {'NumPy db':>12} | {'PyTorch dw':>12} {'PyTorch db':>12} | {'dw diff':>10} {'db diff':>10}")
print("-" * 95)

for step in range(5):
    # NumPy
    dw_np, db_np = compute_gradients(x, y, w_np, b_np)
    
    # PyTorch
    opt.zero_grad()
    y_pred_pt = w_pt * x_t + b_pt
    loss_pt = torch.mean((y_pred_pt - y_t) ** 2)
    loss_pt.backward()
    
    dw_pt = w_pt.grad.item()
    db_pt = b_pt.grad.item()
    
    print(f"{step:4d} | {dw_np:12.6f} {db_np:12.6f} | {dw_pt:12.6f} {db_pt:12.6f} | {abs(dw_np - dw_pt):10.8f} {abs(db_np - db_pt):10.8f}")
    
    # Update both
    w_np -= lr_check * dw_np
    b_np -= lr_check * db_np
    opt.step()

print("\nGradients match at every step!")

---

# Part 9: Exercises

## Exercise 1: Derive and Implement the Gradient for Ridge Regression

The Ridge loss is:

$$\mathcal{L}_{\text{ridge}}(\mathbf{w}) = \frac{1}{N}\|X\mathbf{w} - \mathbf{y}\|^2 + \lambda \|\mathbf{w}\|^2$$

**Task:** 
1. Derive the gradient $\nabla_{\mathbf{w}} \mathcal{L}_{\text{ridge}}$
2. Implement gradient descent for Ridge regression
3. Compare with the closed-form Ridge solution on the multicollinear dataset

In [ ]:
# EXERCISE 1: Ridge Regression Gradient Descent
#
# The gradient of the Ridge loss is:
#   grad = (2/N) * X^T (Xw - y) + 2 * lambda * w
#
# Fill in the function below, then compare with the closed-form solution.

# --- SOLUTION ---

def train_ridge_gd(X, y, lam=0.01, lr=0.001, n_epochs=2000):
    """Gradient descent for Ridge regression."""
    N, p = X.shape
    w = np.zeros(p)
    losses = []
    
    for epoch in range(n_epochs):
        y_pred = X @ w
        error = y_pred - y
        mse = np.mean(error ** 2)
        ridge_penalty = lam * np.sum(w ** 2)
        loss = mse + ridge_penalty
        losses.append(loss)
        
        # Gradient: (2/N) * X^T(Xw - y) + 2*lambda*w
        grad = (2.0 / N) * (X.T @ error) + 2 * lam * w
        w = w - lr * grad
    
    return w, losses

# Test on the multicollinear dataset
lam_test = 1e-4
w_ridge_gd, losses_ridge = train_ridge_gd(X_mc, y_mc, lam=lam_test, lr=0.01, n_epochs=3000)

# Closed-form Ridge solution
N_mc_local = X_mc.shape[0]
w_ridge_cf = np.linalg.solve(
    X_mc.T @ X_mc + N_mc_local * lam_test * np.eye(X_mc.shape[1]),
    X_mc.T @ y_mc
)

print(f"Ridge GD solution:          w = {w_ridge_gd}")
print(f"Ridge closed-form solution: w = {w_ridge_cf}")
print(f"Max absolute difference: {np.max(np.abs(w_ridge_gd - w_ridge_cf)):.6f}")

plt.plot(losses_ridge)
plt.xlabel('Epoch')
plt.ylabel('Ridge Loss')
plt.title('Ridge Regression GD Convergence')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 2: Maximum Safe Learning Rate from Hessian Eigenvalues

**Scenario:** You are given a dataset where the Hessian of the MSE loss has eigenvalues $[0.1, 50]$.

**Tasks:**
1. What is the maximum safe learning rate?
2. What is the condition number?
3. Create a synthetic dataset with these Hessian eigenvalues and verify empirically that GD diverges above the threshold.

In [ ]:
# EXERCISE 2: Hessian eigenvalue analysis

# --- SOLUTION ---

# Given eigenvalues
lam_min, lam_max = 0.1, 50.0

# 1. Maximum safe learning rate
lr_max_theory = 2.0 / lam_max
print(f"1. Max safe lr: 2 / lambda_max = 2 / {lam_max} = {lr_max_theory:.4f}")

# 2. Condition number
kappa = lam_max / lam_min
print(f"2. Condition number: lambda_max / lambda_min = {lam_max} / {lam_min} = {kappa:.1f}")
print(f"   This means convergence will be slow -- the loss landscape is 500x steeper in one direction.")

# 3. Create a synthetic dataset with these eigenvalues
# We need H = (2/N) * X^T X to have eigenvalues [0.1, 50]
# So X^T X should have eigenvalues [0.1*N/2, 50*N/2]
np.random.seed(42)
N_ex2 = 200

# Create X with specific singular values
# If X = U * S * V^T, then X^T X = V * S^2 * V^T
# We want eigenvalues of (2/N)*X^TX to be [0.1, 50]
# So eigenvalues of X^TX should be [0.1*N/2, 50*N/2] = [10, 5000]
# Singular values of X should be [sqrt(10), sqrt(5000)]

# Generate random orthogonal matrix
U, _, _ = np.linalg.svd(np.random.randn(N_ex2, 2), full_matrices=False)
V = np.eye(2)  # Simple choice
S = np.diag([np.sqrt(N_ex2 * 50 / 2), np.sqrt(N_ex2 * 0.1 / 2)])

X_ex2 = U @ S @ V.T
X_ex2_aug = np.column_stack([X_ex2])  # No bias for simplicity

# Verify eigenvalues
H_ex2 = (2.0 / N_ex2) * (X_ex2_aug.T @ X_ex2_aug)
eig_ex2 = np.linalg.eigvalsh(H_ex2)
print(f"\n3. Constructed Hessian eigenvalues: {eig_ex2}")
print(f"   Target: [0.1, 50.0]")

# True weights and targets
w_true_ex2 = np.array([1.0, 2.0])
y_ex2 = X_ex2_aug @ w_true_ex2 + np.random.randn(N_ex2) * 0.5

# Test convergence at various learning rates
test_lrs_ex2 = [0.01, 0.03, lr_max_theory * 0.95, lr_max_theory * 1.05]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, lr_val in zip(axes, test_lrs_ex2):
    w_test, losses_test = train_multivariate_gd(X_ex2_aug, y_ex2, lr=lr_val, n_epochs=100)
    ax.plot(losses_test, linewidth=2)
    ax.set_title(f'lr = {lr_val:.4f}', fontsize=10)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.grid(True, alpha=0.3)
    if any(l > 1e6 for l in losses_test if np.isfinite(l)):
        ax.set_ylim(0, max(100, losses_test[0] * 2))
        ax.text(0.5, 0.9, 'DIVERGES!', transform=ax.transAxes,
                ha='center', fontsize=14, color='red', fontweight='bold')

plt.suptitle(f'Empirical verification: lr_max = {lr_max_theory:.4f}', fontsize=14)
plt.tight_layout()
plt.show()

## Exercise 3: Closed-Form vs GD -- Timing Comparison

**Task:** Create a dataset with 5000 samples and 1000 features. Time both the closed-form solution and gradient descent. Which is faster? At what problem size does GD become preferable?

In [ ]:
# EXERCISE 3: Timing comparison

# --- SOLUTION ---

np.random.seed(42)

feature_sizes = [10, 50, 100, 500, 1000]
N_timing = 5000

print(f"{'p (features)':>12} | {'Closed-form (ms)':>18} | {'GD 500 iters (ms)':>18} | {'Faster':>10}")
print("-" * 75)

for p in feature_sizes:
    X_time = np.random.randn(N_timing, p)
    w_true_time = np.random.randn(p)
    y_time = X_time @ w_true_time + np.random.randn(N_timing) * 0.5
    
    # Time closed-form
    t0 = time.perf_counter()
    for _ in range(5):  # average over 5 runs
        w_cf = np.linalg.solve(X_time.T @ X_time, X_time.T @ y_time)
    t_cf = (time.perf_counter() - t0) / 5 * 1000
    
    # Time GD (500 iterations)
    t0 = time.perf_counter()
    for _ in range(5):
        w_gd_time, _ = train_multivariate_gd(X_time, y_time, lr=0.001, n_epochs=500)
    t_gd = (time.perf_counter() - t0) / 5 * 1000
    
    faster = 'Closed-form' if t_cf < t_gd else 'GD'
    print(f"{p:12d} | {t_cf:18.2f} | {t_gd:18.2f} | {faster:>10}")

print(f"\nNote: closed-form is O(p^3 + Np^2), GD is O(n_epochs * Np).")
print(f"For small p, closed-form wins. For large p, GD wins.")
print(f"Also: GD can work in streaming/online settings; closed-form cannot.")

---

# Summary

## What we learned today

1. **Linear regression** is the simplest supervised learning model: $\hat{y} = wx + b$
2. **MSE loss** measures prediction quality. Its gradient tells us how to improve.
3. **The Normal Equation** gives the closed-form solution: $\hat{w} = (X^TX)^{-1}X^Ty$. It is fast for small $p$ but breaks with multicollinearity.
4. **MSE = Maximum Likelihood** under Gaussian noise. Different noise assumptions lead to different loss functions.
5. **Gradient descent** iteratively updates parameters in the direction that reduces loss.
6. **Three variants:**
   - Batch GD: smooth, expensive per step
   - SGD: noisy, cheap per step
   - Mini-batch: the practical default
7. **Momentum** accelerates convergence by accumulating past gradients.
8. **Learning rate** is critical: too small = slow, too large = diverge. The Hessian eigenvalues tell us the exact threshold.
9. **Condition number** of the Hessian determines convergence speed. Feature scaling improves it.
10. **Ridge regression** (L2 regularization) fixes ill-conditioned problems by making $X^TX$ invertible.
11. **PyTorch autograd** computes the same gradients automatically -- no manual derivation needed.
12. The **canonical PyTorch training loop:** `zero_grad -> forward -> loss -> backward -> step`

## The pattern that drives all of ML

Every model we build for the rest of this course follows the same loop:

```
forward pass -> compute loss -> compute gradients -> update parameters
```

The model gets more complex (CNNs, transformers, etc.), the loss function changes, but the optimization loop stays the same.

## Next session (Session 3)

**Logistic Regression, Evaluation & Data Leakage** -- We change the output from continuous to binary, swap MSE for cross-entropy, and introduce precision/recall/F1. Same optimization ideas, different loss function. Plus: our first Build-Break-Fix exercise.

## Support material

- **Required:** Andrew Ng: Supervised ML -- Coursera week 1 (~1h). StatQuest: Linear Regression (15 min).
- *Optional:* 3Blue1Brown: Gradient Descent (~20 min). Boyd & Vandenberghe: Convex Optimization, Ch. 9 (for convergence theory).

## Homework

Extend to multivariate linear regression (2 features). NumPy from scratch, then PyTorch refactor. Due before Session 4 (Wed Mar 11). See `homework.md` for details.